Cargar las fuentes originales

In [13]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

peliculas = pd.read_csv(
    DATA_DIR / "netflix_movies_detailed_up_to_2025.csv"
)

series = pd.read_csv(
    DATA_DIR / "netflix_tv_shows_detailed_up_to_2025.csv"
)

print("Películas originales:", peliculas.shape)
print("Series originales:", series.shape)
print("Directorio de salida:", PROCESSED_DIR)


Películas originales: (16000, 18)
Series originales: (16000, 16)
Directorio de salida: c:\Users\nayen\EP2-VD-002V-Caso-StreamView-Analytics\data\processed


Depurar las series

In [14]:

# Copias independientes: nunca modificamos las fuentes originales.
series_limpias = series.copy()

# Comprobar que los registros con el mismo ID coincidan
# en todas las columnas, excepto popularity.
columnas_comparacion = [
    c for c in series_limpias.columns
    if c not in ["show_id", "popularity"]
]

grupos = series_limpias[
    series_limpias["show_id"].duplicated(keep=False)
].groupby("show_id")

inconsistencias = []

for show_id, grupo in grupos:
    if (grupo[columnas_comparacion].nunique(dropna=False) > 1).any():
        inconsistencias.append(show_id)

if inconsistencias:
    raise ValueError(
        f"Hay IDs con diferencias adicionales: {inconsistencias}"
    )

filas_antes = len(series_limpias)

# Consolidar cada título por identificador.
agregaciones = {
    columna: "first"
    for columna in series_limpias.columns
    if columna != "popularity"
}
agregaciones["popularity"] = "mean"

series_limpias = (
    series_limpias
    .groupby("show_id", as_index=False, sort=False)
    .agg(agregaciones)
)

print("Filas originales:", filas_antes)
print("Filas después de consolidar:", len(series_limpias))
print("Filas reducidas:", filas_antes - len(series_limpias))
print("IDs únicos:", series_limpias["show_id"].is_unique)


Filas originales: 16000
Filas después de consolidar: 15991
Filas reducidas: 9
IDs únicos: True


Preparar las películas

In [15]:

peliculas_limpias = peliculas.copy()

print("Filas originales:", len(peliculas))
print("Filas procesadas:", len(peliculas_limpias))
print("Duraciones nulas:", peliculas_limpias["duration"].isna().sum())
print(
    "Porcentaje de duración nula:",
    round(peliculas_limpias["duration"].isna().mean() * 100, 2),
    "%"
)

print("Nulos en géneros:", peliculas_limpias["genres"].isna().sum())

# No imputamos duración ni reemplazamos ceros financieros.
# Ambas decisiones requieren más evidencia sobre la fuente.


Filas originales: 16000
Filas procesadas: 16000
Duraciones nulas: 16000
Porcentaje de duración nula: 100.0 %
Nulos en géneros: 107


In [16]:

# Validaciones de integridad
assert peliculas_limpias["show_id"].is_unique
assert series_limpias["show_id"].is_unique
assert len(peliculas_limpias) == len(peliculas)
assert not series_limpias["show_id"].duplicated().any()

# Guardar datasets procesados sin sobrescribir los originales.
peliculas_limpias.to_csv(
    PROCESSED_DIR / "peliculas_procesadas.csv",
    index=False
)

series_limpias.to_csv(
    PROCESSED_DIR / "series_procesadas.csv",
    index=False
)

print("Archivos procesados guardados correctamente.")
print("Películas:", peliculas_limpias.shape)
print("Series:", series_limpias.shape)


Archivos procesados guardados correctamente.
Películas: (16000, 18)
Series: (15991, 16)


In [17]:

# =====================================================
# AUDITORÍA FINAL: DURACIÓN DE PELÍCULAS
# =====================================================

print("=== CALIDAD DEL ARCHIVO ORIGINAL ===")
print("Dimensiones:", peliculas.shape)

resumen_calidad = pd.DataFrame({
    "tipo_dato": peliculas.dtypes.astype(str),
    "valores_nulos": peliculas.isna().sum(),
    "porcentaje_nulos": (
        peliculas.isna().mean() * 100
    ).round(2),
    "valores_unicos": peliculas.nunique()
})

display(
    resumen_calidad.sort_values(
        "porcentaje_nulos",
        ascending=False
    )
)

print("\n=== DIAGNÓSTICO DE DURATION ===")

duration = peliculas["duration"]

print("Tipo de dato:", duration.dtype)
print("Total de filas:", len(duration))
print("Valores nulos:", duration.isna().sum())
print("Valores no nulos:", duration.notna().sum())
print(
    "Porcentaje nulo:",
    round(duration.isna().mean() * 100, 2),
    "%"
)

print("\n=== EJEMPLOS DE PELÍCULAS ===")

display(
    peliculas[
        ["show_id", "title", "release_year", "duration"]
    ].head(15)
)

# Verificar si la columna está completamente vacía.
if duration.isna().all():
    print(
        "\nCONCLUSIÓN: duration está completamente vacía "
        "en la fuente original."
    )
    print(
        "No se imputarán duraciones sin una fuente "
        "complementaria verificable."
    )
else:
    print(
        "\nCONCLUSIÓN: existen duraciones disponibles. "
        "Se debe revisar su distribución y calidad."
    )


=== CALIDAD DEL ARCHIVO ORIGINAL ===
Dimensiones: (16000, 18)


,tipo_dato,valores_nulos,porcentaje_nulos,valores_unicos
duration,float64,16000,100.00,0
country,str,466,2.91,1463
cast,str,204,1.27,15639
director,str,132,0.82,9508
description,str,132,0.82,15854
genres,str,107,0.67,2768
type,str,0,0.00,1
show_id,int64,0,0.00,16000
title,str,0,0.00,15485
date_added,str,0,0.00,4423



=== DIAGNÓSTICO DE DURATION ===
Tipo de dato: float64
Total de filas: 16000
Valores nulos: 16000
Valores no nulos: 0
Porcentaje nulo: 100.0 %

=== EJEMPLOS DE PELÍCULAS ===


,show_id,title,release_year,duration
0,10192,Shrek Forever After,2010,NaN
1,27205,Inception,2010,NaN
2,12444,Harry Potter and the Deathly Hallows: Part 1,2010,NaN
3,38757,Tangled,2010,NaN
4,10191,How to Train Your Dragon,2010,NaN
5,11324,Shutter Island,2010,NaN
6,38575,The Karate Kid,2010,NaN
7,10138,Iron Man 2,2010,NaN
8,38365,Grown Ups,2010,NaN
9,48650,Room in Rome,2010,NaN



CONCLUSIÓN: duration está completamente vacía en la fuente original.
No se imputarán duraciones sin una fuente complementaria verificable.


In [18]:

# =====================================================
# VERIFICAR ARCHIVOS PROCESADOS
# =====================================================

peliculas_guardadas = pd.read_csv(
    PROCESSED_DIR / "peliculas_procesadas.csv"
)

series_guardadas = pd.read_csv(
    PROCESSED_DIR / "series_procesadas.csv"
)

print("\n=== VALIDACIÓN DE ARCHIVOS GUARDADOS ===")
print("Películas guardadas:", peliculas_guardadas.shape)
print("Series guardadas:", series_guardadas.shape)

print(
    "Duraciones nulas en archivo procesado:",
    peliculas_guardadas["duration"].isna().sum()
)

print(
    "IDs únicos en películas:",
    peliculas_guardadas["show_id"].is_unique
)

print(
    "IDs únicos en series:",
    series_guardadas["show_id"].is_unique
)



=== VALIDACIÓN DE ARCHIVOS GUARDADOS ===
Películas guardadas: (16000, 18)
Series guardadas: (15991, 16)
Duraciones nulas en archivo procesado: 16000
IDs únicos en películas: True
IDs únicos en series: True
